In [1]:
from obspy import *
from obspy.signal.cross_correlation import correlation_detector
import pandas as pd
import matplotlib.pyplot as plt
from obspy.clients.fdsn import Client
import matplotlib.dates as mdates
import numpy as np
import datetime
import seaborn as sns

In [2]:
#Small slices: This is a visual check to see if the picker is selecting what want it to
#%run vnda_picker_orig.py 2025-11-24T00:11:30 2025-11-24T00:12:30


In [3]:
#Running the picker on the Philipines earthquake mww7.8. It says there are no earthquakes during the time period so obviously something is not working. Spectrogram looks different than some of the big ones" on the day plots
#%run vnda_picker_orig.py 2026-06-08T11:20 2026-06-08T11:40:00
#st.write('phil_earth_aftershock.mseed', format = 'MSEED')


## Definitions ##

In [4]:
def stat_plotter(df, df1):
    #Calculate hourly/daily metrics
    hourly_counts = df.groupby(df['local'].dt.hour).size() #in local time
    hour_utc = df.groupby(df['onset'].dt.hour).size() #UTC
    daily_counts = df.groupby(df['local'].dt.date).size() #in local time
    daily_utc = df.groupby(df['onset'].dt.date).size()
    
    #Checking if the two dfs are the same
    #Plot hourly variation of signal detection
    fig, ax = plt.subplots()
    ax.scatter(df['hour_local'].unique(), hourly_counts, label = 'unfiltered') #local time
    ax.set_xlabel('Hours in Local Time (UTC+13)')
    ax.set_ylabel('Number of Triggers')
    #ax.set_ylim(50,200)
    ax.scatter(df1['hour_local'].unique(), df1.groupby(df1['local'].dt.hour).size(), color = 'red', label = 'Filtered for double count') #local time
    ax.legend()
    fig.suptitle('Triggers Grouped by Time of Day unfiltered vs filtered')
    
    #plt.show()
    #this was a check to make sure they look the same and not clipping anything. Looks good. 
    
    #fig, ax = plt.subplots()
    #ax.scatter(df['hour_UTC'].unique(), hour_utc) #UTC time
    #ax.set_xlabel('Hours in UTc Time')
    #ax.set_ylabel('Number of Triggers')
    #ax.set_title('Triggers Grouped by Time of Day (UTC)')
    #plt.show()
    
    
    #pick_5 = daily_utc/(24*60) #event time put y-axis in pick per 5 ish min, or pick per minute
    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc/(24), label = 'unfiltered') #pick per hour
    ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), df1.groupby(df1['onset'].dt.date).size()/24, color = 'red', label = 'Filtered for double count')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_xlabel('Days in UTC Time')
    #ax.set_ylim(5,15)
    ax.set_title("Average Number of Events per Hour")
    ax.set_ylabel('Number of Triggers per Hour')
    ax.legend()
    
    fig.tight_layout(pad=3.0) 

    fig.add_gridspec(4, 4, wspace=0, hspace=0)
   
    #interevent time plot
   
    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.hist(df['interevent_time'].dt.total_seconds()/60, bins = 80, edgecolor = 'black', label = 'unfiltered') 
    ax.hist(df1['interevent_time'].dt.total_seconds()/60, bins = 80, edgecolor = 'black', color = 'red', label = 'filtered')
    #ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_xlabel('Time in Minutes')
    ax.set_title("Interevent Spacing")
    ax.set_ylabel('Occurances')
    ax.legend()
    plt.show()

    fig, ax = plt.subplots()

    #ax = df.groupby(pd.Grouper(key='onset', freq='h')).size().plot(color = 'blue')
    hourly_counts = df1.groupby(pd.Grouper(key='onset', freq='h')).size()
    print(type(hourly_counts))
    ax = hourly_counts.plot(color = 'red', label = 'filtered')
    ax.set_xlabel('Days in UTC Time')
    ax.set_title("Number of Events per Hour")
    ax.set_ylabel('Triggers')
    #ax.set_ylim(0, 25)
    fig.tight_layout(pad=3.0) 
    fig.legend()
    fig.add_gridspec(8, 8, wspace=0, hspace=0)

    return None
    

In [5]:
def graphs(df):
    
    #Calculate hourly/daily metrics
    hourly_counts = df.groupby(df['local'].dt.hour).size() #in local time
    hour_utc = df.groupby(df['onset'].dt.hour).size() #UTC
    daily_counts = df.groupby(df['local'].dt.date).size() #in local time
    daily_utc = df.groupby(df['onset'].dt.date).size()

    #Average values
    #unfilt_min = trigger_df['interevent_time'].dt.total_seconds()/60
    filt_min = df['interevent_time'].dt.total_seconds()/60
    daily_mean = (daily_utc/24).mean()
    
    #Plot hourly variation of signal detection
    fig, ax = plt.subplots()
    ax.scatter(df['hour_local'].unique(), hourly_counts, label = 'unfiltered') #local time
    ax.set_xlabel('Hours in Local Time (UTC+13)')
    ax.set_ylabel('Number of Triggers')
    ax.set_ylim(None,None)
    #ax.scatter(df1['hour_local'].unique(), df1.groupby(df1['local'].dt.hour).size(), color = 'red', label = 'Filtered for double count') #local time
    ax.legend()
    fig.suptitle('Triggers Grouped by Time of Day unfiltered vs filtered')
    
    #Plot Average Number of Events per Hour
    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc/(24), label = 'trigger counts', color = 'purple', edgecolor = 'black') #pick per hour
    
    ax.axhline(y = round(daily_mean,2), color = 'black', linestyle = '--', linewidth = 2, label = f'{round(daily_mean,2)}, triggers', alpha = 0.7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_xlabel('Days in UTC Time')
    ax.set_ylim(5,12)
    ax.set_title("Average Number of Events per Hour")
    ax.set_ylabel('Triggers per Hour')
    ax.legend()
    #plt.savefig(f"Number_events_{filtered_df['local'].dt.date.min()}_to_{filtered_df['local'].dt.date.max()}",dpi=200)

    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.hist(df['interevent_time'].dt.total_seconds()/60, bins = 80, edgecolor = 'black', label = 'trigger counts')
    ax.axvline(x = round(filt_min.mean(),2), color = 'black', linestyle = '--', linewidth = 2, label = f'{round(filt_min.mean(),2)} minutes', alpha = 0.7)
    ax.set_xlabel('Time in Minutes')
    ax.set_title("Interevent Spacing")
    ax.set_ylabel('Occurances')
    ax.legend()
    #plt.show()
    plt.savefig(f"Intervent_Time_{filtered_df['local'].dt.date.min()}_to_{filtered_df['local'].dt.date.max()}",dpi=200)
    
    
    fig, ax = plt.subplots()

    #ax = df.groupby(pd.Grouper(key='onset', freq='h')).size().plot(color = 'blue')
    hourly_counts = df.groupby(pd.Grouper(key='onset', freq='h')).size()
    ax = hourly_counts.plot(color = 'red', label = 'trigger counts')
    ax.set_xlabel('Days in UTC Time')
    ax.set_title("Number of Events per Hour")
    ax.set_ylabel('Triggers')
    ax.set_ylim(0, 25)
    fig.tight_layout(pad=3.0) 
    fig.legend()
    fig.add_gridspec(8, 8, wspace=0, hspace=0)
    return None

In [6]:
def triggers_func(trig_times):
    trigger_df = pd.DataFrame(trig_times)
    #trigger_df.set_index()
    trigger_df[0] = [t.datetime for t in trigger_df[0]]
    trigger_df[1] = [t.datetime for t in trigger_df[1]]
    trigger_df['month'] = trigger_df[0].dt.month
    trigger_df['day'] = trigger_df[0].dt.day
    trigger_df['hour_UTC'] = trigger_df[0].dt.hour
    
    #Times in local Antarctica time, better for daylight tracking, diurnal patterns
    trigger_df['local'] = trigger_df[0].dt.tz_localize('UTC').dt.tz_convert('Antarctica/McMurdo')
    trigger_df['hour_local'] = trigger_df['local'].dt.hour
    trigger_df['day_local'] = trigger_df['local'].dt.day
    trigger_df.rename(columns={0:'onset', 1:'offset'}, inplace = True)
    
    #Interevent spacing
    trigger_df['interevent_time'] = trigger_df['onset'].shift(-1) - trigger_df['offset']
    
    #Hourly and Daily counts
    trigger_df["hourly_counts"] = trigger_df.groupby(trigger_df['local'].dt.hour).size() #in local time
    hour_utc = trigger_df.groupby(trigger_df['onset'].dt.hour).size() #UTC
    daily_counts = trigger_df.groupby(trigger_df['local'].dt.day).size() #in local time
    daily_utc = trigger_df.groupby(trigger_df['onset'].dt.day).size()
    trigger_df['amplitude m'] = pd.DataFrame(peak_amps) #amplitude - displacement in meters
    
    #Filtered to no events under 1 minute 
    filtered_df = trigger_df[trigger_df['interevent_time'].dt.total_seconds() > 60]
    
    return trigger_df, filtered_df

In [7]:
def interevent_spacing(df):
    '''
    Definition for plotting a histogram of interevent spacing. 
    Option to group by month. 
    '''
    num_months = int(df['month'].nunique())
    #filt_min = filtered_df['interevent_time'].dt.total_seconds()/60
    print(f'there are {num_months} months')
    fig, ax = plt.subplots(num_months+1, figsize=(10, 8))
    for i , month in enumerate(df['month'].unique(), start = 0):
        #month = trigger_df[trigger_df.index.month == month]
        #interevent_spacing = trigger_df['interevent_time'].dt.total_seconds()/60
        print(month)
        data = df[df['onset'].dt.month == month][['interevent_time']]
        data.dropna()
        a = data['interevent_time'].dt.total_seconds()/60
        filt_min = round(data.mean(),2)
        color = sns.color_palette()
        #print(data), print(type(data))
        #monthly_triggers = trigger_df.groupby(trigger_df['onset'].dt.to_period('M'))['interevent_time']
        
        #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
        ax[i].hist(a, bins = 80, edgecolor = 'black', label = f'trigger counts for month {month}', color = color[i])
        ax[i].axvline(x = filt_min.iloc[0].total_seconds()/60, color = 'black', linestyle = '--', linewidth = 2, label = f'{round(filt_min.iloc[0].total_seconds()/60,2)} minutes')
        sns.kdeplot(a,color = color[i], ax = ax[-1], label = month)
        ax[i].legend()
        ax[i].set_xlim(0, 45)
        ax[-1].set_xlim(0, 45)
        ax[i].set_xticklabels([])
        ax[i].set_xticks([])
    fig.supxlabel('Time in Minutes')
    
    fig.tight_layout()
    fig.suptitle("Interevent Spacing", ha='center', va='top')
    fig.subplots_adjust(top=0.88)
    fig.supylabel('Occurances')
    #fig.suplegend()
    plt.show()
    return None

## Run the picker ##

In [ ]:
#Call vnda_picker_orig.py with the associated date range. This will output a list of trigger times. 
#I have the plottnig function of this set to False. Change to true to see the waveform/spectrogram
#CHANGE plot to FALSE otherwise it will be super messy
%run vnda_picker_orig.py 2025-01-01T00:00:00 2026-01-01T00:00:00


/Users/kitsellusted/miniconda3/envs/pygmt-env/lib/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)
/Users/kitsellusted/grad_school/VANDA_work/vnda_picker_orig.py:50: ObsPyDeprecationWarning: attach_response is deprecated and will be removed in a future release. Use remove_response() instead.
  st = client2.get_waveforms(


In [ ]:
#Calculating triggers
trigger_df, filtered_df = triggers_func(trig_times)

In [ ]:
#Some stats on the difference between the filtered and unfiltered data. 
unfilt_min = trigger_df['interevent_time'].dt.total_seconds()/60
filt_min = filtered_df['interevent_time'].dt.total_seconds()/60
print('unfiltered average interevent spacing:', round(unfilt_min.mean(),2), 'minutes')
#print('unfiltered std interevent spacing:', round(unfilt_min.std(),3), 'minutes')
#print('unfiltered mode interevent spacing:', round(unfilt_min.mode(),3), 'minutes')

print('filtered average interevent spacing:', round(filt_min.mean(),2), 'minutes')
#print('filtered std interevent spacing:', round(filt_min.std(),3), 'minutes')
#print('filtered mode interevent spacing:', round(filt_min.mode(),3), 'minutes')


In [ ]:
interevent_spacing(filtered_df)

In [ ]:
sys.exit()
# COMPARING THE FILTERED VS. UNFILTERED TRIGGER PICKS. FILTERING BY TIME INBETWEEN TRIGGERS
#adding in the additional filter by amplitude reduces the likelihood of double picking!
stat_plotter(trigger_df, filtered_df)
#graphs(filtered_df)

In [ ]:
graphs(filtered_df)

In [ ]:
sys.exit()
#Dayplots
if st[0].stats.endtime - st[0].stats.starttime >= 24*60*60:
    for tr in st:
        for i in range(len(pd.to_datetime(trigger_df['onset']).dt.date.unique())):
            start_time = st[0].stats.starttime
            day_start = start_time + (i * 86400)
            day_end = day_start + 86400
            daily_st = st.slice(day_start, day_end)
    
            local_offset = +13 * 3600
            daily_st.plot(type='dayplot', interval=60, tick_format='%m/%d %Hh', 
                          offset=local_offset,
                          show_y_UTC_label=False  )# interval is minutes per line
else:
    print('stream is less than 1 day. this will not work')
            
            


In [ ]:
#%run dayplot.py # ok this doesn't work right now. 

In [ ]:
#%run vnda_picker_orig.py 2025-11-21T00:00 2025-11-29T00:00:00

In [ ]:
#Overlaid waveforms

print(len(sliced_wvf))
i = 0
fig,ax = plt.subplots(figsize=(10,5))
fig.suptitle('VNDA Signals Overlaid')
fig.supxlabel('Time (s)')
fig.supylabel('Displacement (m)')
time = sliced_wvf[0]
for i, trace in enumerate(sliced_wvf):
    cmap = plt.colormaps['gist_heat']
    #print(tr)
    #trace.normalize()
    color_list = [cmap(i) for i in np.linspace(0, 1, len(sliced_wvf))]
    ax.plot(trace.times(), trace.data, color=color_list[i], alpha = 0.6, label='Original')
    i +=1 

